In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
working_directory = Path.cwd().resolve()

project_root = next(
    (
        path
        for path in (working_directory, *working_directory.parents)
        if (path / "src" / "procesado" / "limpiar_poblacion.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project root containing src/procesado.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 1. Cleaning population

Read the Andalusian population Bronze table, apply the current cleaning steps, and preview the result.

In [2]:
from src.procesado.limpiar_poblacion import clean_population
from src.procesado.paths import (
    POPULATION_BRONZE_PATH,
    POPULATION_FAILED_PATH,
    POPULATION_SILVER_PATH,
)

population_bronze = pd.read_csv(
    POPULATION_BRONZE_PATH,
    dtype=str,
    keep_default_na=False,
)
population_cleaned = clean_population(population_bronze)

POPULATION_SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
population_cleaned.to_parquet(POPULATION_SILVER_PATH, index=False)
population_failed = pd.read_parquet(POPULATION_FAILED_PATH)
print(
    f"Bronze rows: {len(population_bronze):,}; "
    f"cleaned rows: {len(population_cleaned):,}; "
    f"failed rows: {len(population_failed):,}; "
    f"saved to: {POPULATION_SILVER_PATH}"
)
print(f"Rejected rows saved to: {POPULATION_FAILED_PATH}")
display(population_cleaned.head(10))

Bronze rows: 1,201,050; cleaned rows: 3,925; failed rows: 0; saved to: C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\processed\silver\population_andalusia.parquet
Rejected rows saved to: C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\processed\silver\failed\population_failed.parquet


,province_code,province,municipality_code,municipality_name,year,population_total,population_18_50,source,downloaded_at_utc
0,04,Almería,04001,Abla,2021,1260,479,INE,2026-10-19 07:07:30.076995+00:00
1,04,Almería,04001,Abla,2022,1248,474,INE,2026-10-19 07:07:30.076995+00:00
2,04,Almería,04001,Abla,2023,1259,474,INE,2026-10-19 07:07:30.076995+00:00
3,04,Almería,04001,Abla,2024,1277,480,INE,2026-10-19 07:07:30.076995+00:00
4,04,Almería,04001,Abla,2025,1267,464,INE,2026-10-19 07:07:30.076995+00:00
5,04,Almería,04002,Abrucena,2021,1212,461,INE,2026-10-19 07:07:30.076995+00:00
6,04,Almería,04002,Abrucena,2022,1220,464,INE,2026-10-19 07:07:30.076995+00:00
7,04,Almería,04002,Abrucena,2023,1274,475,INE,2026-10-19 07:07:30.076995+00:00
8,04,Almería,04002,Abrucena,2024,1234,442,INE,2026-10-19 07:07:30.076995+00:00
9,04,Almería,04002,Abrucena,2025,1235,442,INE,2026-10-19 07:07:30.076995+00:00


# 2. Cleaning income

Read the INE Bronze table, keep per-person income, save the Silver Parquet, and preview the result.

In [3]:
from src.procesado.limpiar_renta import clean_income
from src.procesado.paths import (
    INCOME_BRONZE_PATH,
    INCOME_FAILED_PATH,
    INCOME_SILVER_PATH,
)

income_bronze = pd.read_csv(
    INCOME_BRONZE_PATH,
    dtype=str,
    keep_default_na=False,
)
income_cleaned = clean_income(income_bronze)

INCOME_SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
income_cleaned.to_parquet(INCOME_SILVER_PATH, index=False)
print(
    f"Bronze rows: {len(income_bronze):,}; "
    f"cleaned rows: {len(income_cleaned):,}; "
    f"failed rows: {len(pd.read_parquet(INCOME_FAILED_PATH)):,}; "
    f"saved to: {INCOME_SILVER_PATH}"
)
display(income_cleaned.head(10))

Bronze rows: 786; cleaned rows: 783; failed rows: 3; saved to: C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\processed\silver\income_andalusia.parquet


,province_code,province,municipality_code,municipality_name,year,mean_net_income_per_person,source,downloaded_at_utc
0,04,Almería,04001,Abla,2023,12411,INE,2026-10-19 07:09:25.321278+00:00
1,04,Almería,04002,Abrucena,2023,11944,INE,2026-10-19 07:09:25.321278+00:00
2,04,Almería,04003,Adra,2023,10156,INE,2026-10-19 07:09:25.321278+00:00
3,04,Almería,04004,Albanchez,2023,12035,INE,2026-10-19 07:09:25.321278+00:00
4,04,Almería,04005,Alboloduy,2023,11557,INE,2026-10-19 07:09:25.321278+00:00
5,04,Almería,04006,Albox,2023,10742,INE,2026-10-19 07:09:25.321278+00:00
6,04,Almería,04007,Alcolea,2023,11126,INE,2026-10-19 07:09:25.321278+00:00
7,04,Almería,04008,Alcóntar,2023,13927,INE,2026-10-19 07:09:25.321278+00:00
8,04,Almería,04009,Alcudia de Monteagud,2023,13871,INE,2026-10-19 07:09:25.321278+00:00
9,04,Almería,04010,Alhabia,2023,10670,INE,2026-10-19 07:09:25.321278+00:00


# 3. Cleaning gyms

Read the Andalusian OSM Bronze gym table, clean it, save the GeoParquet Silver table, and preview the result.

In [4]:
from src.procesado.limpiar_gimnasios import clean_gyms
from src.procesado.paths import (
    GYMS_BRONZE_PATH,
    GYMS_FAILED_PATH,
    GYMS_SILVER_PATH,
)

gyms_bronze = pd.read_csv(
    GYMS_BRONZE_PATH,
    dtype=str,
    keep_default_na=False,
)
gyms_cleaned = clean_gyms(gyms_bronze)

GYMS_SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
gyms_cleaned.to_parquet(GYMS_SILVER_PATH, index=False)
print(
    f"Bronze rows: {len(gyms_bronze):,}; "
    f"Silver rows: {len(gyms_cleaned):,}; "
    f"gyms: {gyms_cleaned['sport_centre_type'].eq('gym').sum():,}; "
    f"other sports facilities: "
    f"{gyms_cleaned['sport_centre_type'].eq('other_sport_facility').sum():,}; "
    f"failed rows: {len(pd.read_parquet(GYMS_FAILED_PATH)):,}"
)
print(f"Silver saved to: {GYMS_SILVER_PATH}")
display(gyms_cleaned.head(10))

Bronze rows: 3,567; Silver rows: 3,567; gyms: 564; other sports facilities: 3,003; failed rows: 0
Silver saved to: C:\Users\fatima.d.villalba\OneDrive - Accenture\Desktop\repos\market-expansion-sot\data\processed\silver\gyms_andalusia.parquet


,osm_type,osm_id,province_code,province,latitude,longitude,source,downloaded_at_utc,tag_name,tag_sport,sport_centre_type,geometry
0,node,260812441,ES-AL,Almería,37.172028,-1.833994,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Go-Karting,karting,other_sport_facility,POINT (-1.83399 37.17203)
1,node,1620487480,ES-AL,Almería,37.345363,-2.432541,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Pabellón,<NA>,other_sport_facility,POINT (-2.43254 37.34536)
2,node,1912986759,ES-AL,Almería,37.212201,-1.810827,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Vera Costa,fitness,other_sport_facility,POINT (-1.81083 37.2122)
3,node,2452253961,ES-AL,Almería,36.838087,-2.949279,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Pabellon de deportes,<NA>,other_sport_facility,POINT (-2.94928 36.83809)
4,node,4432638479,ES-AL,Almería,36.969974,-2.607533,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Frontón de Huécija,pelota,other_sport_facility,POINT (-2.60753 36.96997)
5,node,4512584749,ES-AL,Almería,36.838962,-2.460986,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,e-fitness,<NA>,gym,POINT (-2.46099 36.83896)
6,node,4512664045,ES-AL,Almería,36.839466,-2.456743,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,BOX GYM,<NA>,gym,POINT (-2.45674 36.83947)
7,node,4708078896,ES-AL,Almería,36.696758,-2.789563,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Be in Shape,<NA>,gym,POINT (-2.78956 36.69676)
8,node,5319369078,ES-AL,Almería,37.352822,-2.183695,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Gimnasio municipal,<NA>,gym,POINT (-2.18369 37.35282)
9,node,5319369079,ES-AL,Almería,37.352754,-2.183941,OpenStreetMap,2026-10-19 07:10:36.840029+00:00,Pabellón Municipal de Deportes,<NA>,other_sport_facility,POINT (-2.18394 37.35275)
